# 4.12

In [ ]:
#/home/aadithya-iyer/Github/IIScSecondSem/IISC_PRNN/Assignment 1/PRNN_2026_A1_data/dataset_4.csv
#Import the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#Load the dataset
dataset = pd.read_csv('/home/aadithya-iyer/Github/IIScSecondSem/IISC_PRNN/Assignment 1/Data_Codes_Renamed/dataset_4.csv')
#All but last col are features
X = dataset.iloc[:, :-1].values
#Last col is label
y = dataset.iloc[:, -1].values
#Label is binary


In [9]:
#Step 1: Logistic Regression with 1 vs Rest:
from turtle import update


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
#Logistic regression function 1 vs rest
#returns iterations required to converge and weights
def logistic_regression(X, y, num_classes, learning_rate=0.1, max_iterations=1000, tolerance=1e-3):

    m, n = X.shape

    X = np.hstack((np.ones((m,1)), X))

    weights = np.zeros((num_classes, n+1))
    iterations = np.zeros(num_classes)

    for i in range(num_classes):

        y_binary = (y == i).astype(int)

        for iteration in range(max_iterations):

            scores = np.dot(X, weights[i])
            probabilities = sigmoid(scores)

            gradient = np.dot(X.T, (probabilities - y_binary)) / m

            weights[i] -= learning_rate * gradient

            update = learning_rate * gradient
            weights[i] -= update

            if np.linalg.norm(update) < tolerance:
                iterations[i] = iteration
                break
        else:
            iterations[i] = max_iterations

    return weights, iterations
#Predict function for logistic regression
def predict_logistic_regression(X, weights):

    m, n = X.shape
    X = np.hstack((np.ones((m,1)), X))

    scores = np.dot(X, weights.T)
    probabilities = sigmoid(scores)

    return np.argmax(probabilities, axis=1)
#Train logistic regression model on raw data

num_classes = len(np.unique(y))
iterations = np.zeros(num_classes)
weights, iterations = logistic_regression(X, y, num_classes)
if np.any(iterations == 1000):
    print("Warning: Some classes did not converge within the maximum iterations.")
print("Iterations per class (raw):", iterations)
print("Average iterations (raw):", np.mean(iterations))

#Now, implement standardscaler:
#x' = (x - mean) / std
class StandardScaler:

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1

    def transform(self, X):
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)
#Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
#Train logistic regression model on scaled data
iterations_scaled = np.zeros(num_classes)
weights_scaled, iterations_scaled = logistic_regression(X_scaled, y, num_classes)

print("Iterations per class (scaled):", iterations_scaled)
print("Average iterations (scaled):", np.mean(iterations_scaled))
if np.any(iterations_scaled == 1000):
    print("Warning: Some classes did not converge within the maximum iterations.")


Iterations per class (raw): [1000. 1000.]
Average iterations (raw): 1000.0
Iterations per class (scaled): [316. 316.]
Average iterations (scaled): 316.0


# 4.13, 4.14

In [10]:
#KNN Classifier from scratch:
#One for computing pairwise euclidean distances with
#Nested for loops (inefficient):
def knn_predict_loops(X_train, y_train, X_test, k):

    n_test = X_test.shape[0]
    n_train = X_train.shape[0]

    y_pred = []

    for i in range(n_test):

        distances = np.zeros(n_train)

        for j in range(n_train):

            diff = X_train[j] - X_test[i]
            distances[j] = np.sqrt(np.sum(diff**2))

        nearest_indices = np.argsort(distances)[:k]
        nearest_labels = y_train[nearest_indices]

        most_common = np.bincount(nearest_labels).argmax()
        y_pred.append(most_common)

    return np.array(y_pred)
#Now, implement KNN with vectorized distance computation:
def knn_predict_vectorized(X_train, y_train, X_test, k):

    train_sq = np.sum(X_train**2, axis=1)
    test_sq = np.sum(X_test**2, axis=1)

    dists = np.sqrt(
        train_sq[:, None] +
        test_sq[None, :] -
        2 * X_train @ X_test.T
    )

    nearest_indices = np.argsort(dists, axis=0)[:k]

    nearest_labels = y_train[nearest_indices]

    y_pred = np.array([
        np.bincount(nearest_labels[:, i]).argmax()
        for i in range(X_test.shape[0])
    ])

    return y_pred

#Take 2000 sample test split from the dataset
X_train, y_train = X[:8000], y[:8000]
X_test, y_test = X[8000:10000], y[8000:10000]
#Predict with KNN (k=5) on raw data
k = 5
#Calculate speedup of vectorized KNN vs nested loops
import time
start_time = time.time()
y_pred_knn = knn_predict_loops(X_train, y_train, X_test, k)
loop_time = time.time() - start_time

print(f"Nested KNN time: {loop_time:.4f} seconds")


# ---- Benchmark vectorized ----
start_time = time.time()
y_pred_knn_vectorized = knn_predict_vectorized(X_train, y_train, X_test, k)
vec_time = time.time() - start_time

print(f"Vectorized KNN time: {vec_time:.4f} seconds")


# ---- Speedup factor ----
speedup = loop_time / vec_time
print(f"Speedup factor: {speedup:.2f}x")


# ---- Accuracy on RAW data ----
accuracy_knn_raw = np.mean(y_pred_knn_vectorized == y_test)
print(f"KNN Accuracy on raw data: {accuracy_knn_raw:.4f}")


# ---- Scale the dataset ----
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ---- Accuracy on SCALED data ----
y_pred_knn_scaled = knn_predict_vectorized(
    X_train_scaled,
    y_train,
    X_test_scaled,
    k
)

accuracy_knn_scaled = np.mean(y_pred_knn_scaled == y_test)
print(f"KNN Accuracy on scaled data: {accuracy_knn_scaled:.4f}")


# ---- Accuracy difference ----
accuracy_difference = accuracy_knn_scaled - accuracy_knn_raw
print(f"Difference in KNN accuracy (scaled - raw): {accuracy_difference:.4f}")
#Exact difference in accuracy between KNN on raw vs scaled data
accuracy_difference = accuracy_knn_scaled - accuracy_knn_raw
print(f"Difference in KNN accuracy (scaled - raw): {accuracy_difference:.4f}")
#Speedup Factor of vectorized KNN vs nested loops:
speedup = loop_time / vec_time
print(f"Speedup factor: {speedup:.2f}x")

Nested KNN time: 135.4804 seconds
Vectorized KNN time: 1.4350 seconds
Speedup factor: 94.41x
KNN Accuracy on raw data: 0.9045
KNN Accuracy on scaled data: 0.9210
Difference in KNN accuracy (scaled - raw): 0.0165
Difference in KNN accuracy (scaled - raw): 0.0165
Speedup factor: 94.41x


# 4.15

In [12]:
classes = np.unique(y)
num_classes = len(classes)
num_features = X.shape[1]

# Compute prior probabilities
prior_probabilities = np.array([
    np.mean(y == c) for c in classes
])

means = np.zeros((num_classes, num_features))
variances = np.zeros((num_classes, num_features))

for idx, c in enumerate(classes):
    X_c = X[y == c]
    means[idx] = np.mean(X_c, axis=0)
    variances[idx] = np.var(X_c, axis=0) + 1e-9


# ---- Log version ----
def naive_bayes_predict_log(X, prior_probabilities, means, variances):

    n_samples = X.shape[0]
    n_classes = len(prior_probabilities)

    log_posterior = np.zeros((n_samples, n_classes))

    for c in range(n_classes):

        log_prior = np.log(prior_probabilities[c])

        log_likelihood = -0.5 * np.sum(
            np.log(2 * np.pi * variances[c]) +
            ((X - means[c]) ** 2) / variances[c],
            axis=1
        )

        log_posterior[:, c] = log_prior + log_likelihood

    return np.argmax(log_posterior, axis=1)


# Prediction
y_pred_nb = naive_bayes_predict_log(X, prior_probabilities, means, variances)
accuracy_nb = np.mean(y_pred_nb == y)

print(f"Naive Bayes Accuracy (log version): {accuracy_nb:.4f}")

#Now, do it without log probabilities, and report the exact feature dimension
#at which underflow occurs:
def naive_bayes_predict_no_log(X, prior_probabilities, means, variances):

    n_samples = X.shape[0]
    n_classes = len(prior_probabilities)

    posterior = np.zeros((n_samples, n_classes))

    for c in range(n_classes):

        prior = prior_probabilities[c]

        likelihood = np.prod(
            (1 / np.sqrt(2 * np.pi * variances[c])) *
            np.exp(-((X - means[c]) ** 2) / (2 * variances[c])),
            axis=1
        )

        posterior[:, c] = prior * likelihood

    return posterior
#Predict with Naive Bayes without log probabilities
underflow_dim = None

for d in range(1, X.shape[1] + 1):

    posterior = naive_bayes_predict_no_log(
        X[:, :d],
        prior_probabilities,
        means[:, :d],
        variances[:, :d]
    )

    if np.any(posterior == 0):
        underflow_dim = d
        break

print(f"Underflow occurs at feature dimension: {underflow_dim}")

Naive Bayes Accuracy (log version): 0.9133
Underflow occurs at feature dimension: None
